In [3]:
import os

# 1. Nonaktifkan GPU TensorFlow agar tidak bentrok memori C++ dengan MediaPipe
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp

print("=" * 60)
print("     SIMULASI INTEGRASI CRNN - RNN REAL-TIME")
print("=" * 60)

# Informasi Versi
print(f"TensorFlow : {tf.__version__}")
print(f"MediaPipe  : {mp.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"NumPy      : {np.__version__}")

# Konfigurasi Input Model CRNN
SEQUENCE_LENGTH = 30
FEATURES_PER_FRAME = 63  # 21 landmark x 3 (x, y, z)
NUM_CLASSES = 26
EXPECTED_INPUT_SHAPE = (SEQUENCE_LENGTH, FEATURES_PER_FRAME)

print("\nKonfigurasi:")
print(f"- Panjang sequence : {SEQUENCE_LENGTH} frame")
print(f"- Fitur per frame  : {FEATURES_PER_FRAME}")
print(f"- Jumlah kelas     : {NUM_CLASSES}")
print(f"- Input CRNN       : {EXPECTED_INPUT_SHAPE}")

print("\n✓ Konfigurasi simulasi berhasil disiapkan.")
print("=" * 60)

     SIMULASI INTEGRASI CRNN - RNN REAL-TIME
TensorFlow : 2.19.1
MediaPipe  : 0.10.14
OpenCV     : 4.13.0
NumPy      : 2.1.3

Konfigurasi:
- Panjang sequence : 30 frame
- Fitur per frame  : 63
- Jumlah kelas     : 26
- Input CRNN       : (30, 63)

✓ Konfigurasi simulasi berhasil disiapkan.


In [4]:
import mediapipe as mp
# Impor langsung modul solutions dari mediapipe
from mediapipe.python.solutions import hands as mp_hands
from mediapipe.python.solutions import drawing_utils as mp_drawing

print("=" * 60)
print("     INISIALISASI MEDIAPIPE WEBCAM (STABLE API)")
print("=" * 60)

# Inisialisasi Hand Landmarker
webcam_landmarker = mp_hands.Hands(
    static_image_mode=False,     # Mode stream video real-time
    max_num_hands=1,             # Deteksi 1 tangan
    model_complexity=1,          # Kompleksitas model standar
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("✓ MediaPipe Hand Landmarker berhasil dibuat.")
print("✓ Running Mode : Stream Video")
print("✓ Jumlah tangan : 1")
print("✓ Fitur per frame: 63 (21 landmark x 3)")
print("\n✓ MediaPipe siap digunakan untuk pipeline webcam!")
print("=" * 60)

     INISIALISASI MEDIAPIPE WEBCAM (STABLE API)
✓ MediaPipe Hand Landmarker berhasil dibuat.
✓ Running Mode : Stream Video
✓ Jumlah tangan : 1
✓ Fitur per frame: 63 (21 landmark x 3)

✓ MediaPipe siap digunakan untuk pipeline webcam!


I0000 00:00:1787842425.770268    5797 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1787842425.775865   15438 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics (ICL GT1)


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787842425.839036   15426 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787842425.861973   15427 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
# ==========================================
# CELL 3: WEBCAM REAL-TIME
# DETEKSI 21 LANDMARK TANGAN
# ==========================================

print("=" * 60)
print("     MEMULAI WEBCAM REAL-TIME")
print("=" * 60)

# ==========================================
# 1. INISIALISASI WEBCAM
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Webcam tidak dapat dibuka.")
else:
    print("✓ Webcam berhasil dibuka.")
    print("Tekan 'q' pada jendela webcam untuk berhenti.")


# ==========================================
# 2. VARIABEL STATUS
# ==========================================

frame_count = 0

# Jumlah frame yang berhasil mendeteksi tangan
detected_frame_count = 0

# Jumlah landmark pada frame terakhir yang berhasil
last_landmark_count = 0

# Jumlah fitur pada frame terakhir
last_feature_count = 0


# ==========================================
# 3. LOOP REAL-TIME
# ==========================================

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        print("❌ Frame webcam tidak berhasil dibaca.")
        break

    frame_count += 1

    # --------------------------------------
    # Mirror webcam
    # --------------------------------------

    frame = cv2.flip(frame, 1)

    # --------------------------------------
    # BGR → RGB
    # --------------------------------------

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    # --------------------------------------
    # Deteksi tangan
    # --------------------------------------

    results = webcam_landmarker.process(
        frame_rgb
    )

    # ======================================
    # JIKA TANGAN TERDETEKSI
    # ======================================

    if results.multi_hand_landmarks:

        hand_landmarks = (
            results.multi_hand_landmarks[0]
        )

        # Hitung frame yang berhasil
        detected_frame_count += 1

        # Jumlah landmark
        last_landmark_count = len(
            hand_landmarks.landmark
        )

        # ----------------------------------
        # Gambar landmark
        # ----------------------------------

        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )

        # ----------------------------------
        # Ekstraksi koordinat
        # ----------------------------------

        frame_features = []

        for lm in hand_landmarks.landmark:

            frame_features.extend([
                lm.x,
                lm.y,
                lm.z
            ])

        frame_features = np.array(
            frame_features,
            dtype=np.float32
        )

        # Simpan jumlah fitur
        last_feature_count = len(
            frame_features
        )

        # ----------------------------------
        # Informasi pada webcam
        # ----------------------------------

        cv2.putText(
            frame,
            f"Landmarks : {last_landmark_count}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Features  : {last_feature_count}",
            (20, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Tangan TERDETEKSI",
            (20, 110),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    # ======================================
    # JIKA TANGAN TIDAK TERDETEKSI
    # ======================================

    else:

        cv2.putText(
            frame,
            "Tangan tidak terdeteksi",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

        cv2.putText(
            frame,
            "Features  : 0",
            (20, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )


    # ======================================
    # 4. INFORMASI FRAME
    # ======================================

    cv2.putText(
        frame,
        f"Frame : {frame_count}",
        (20, 145),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Detected : {detected_frame_count}",
        (20, 180),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ======================================
    # 5. TAMPILKAN WEBCAM
    # ======================================

    cv2.imshow(
        "SIBI - Real-Time Hand Landmark",
        frame
    )


    # ======================================
    # 6. TOMBOL KELUAR
    # ======================================

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# ==========================================
# 7. MEMBERSIHKAN RESOURCE
# ==========================================

cap.release()
cv2.destroyAllWindows()


# ==========================================
# 8. HASIL VERIFIKASI
# ==========================================

print("\n" + "=" * 60)
print("     HASIL VERIFIKASI WEBCAM")
print("=" * 60)

print(f"Total frame terbaca       : {frame_count}")
print(f"Frame dengan tangan       : {detected_frame_count}")
print(f"Frame tanpa tangan        : {frame_count - detected_frame_count}")
print(f"Landmark/frame terakhir   : {last_landmark_count}")
print(f"Fitur/frame terakhir      : {last_feature_count}")

if detected_frame_count > 0:
    print("\n✓ Webcam berhasil mendeteksi tangan.")
    print("✓ Landmark tangan berhasil diperoleh.")
    
    if last_landmark_count == 21:
        print("✓ Jumlah landmark sesuai: 21.")
    
    if last_feature_count == 63:
        print("✓ Jumlah fitur sesuai: 63.")
        
else:
    print("\n❌ Tidak ada tangan yang berhasil dideteksi.")

print("=" * 60)

     MEMULAI WEBCAM REAL-TIME
✓ Webcam berhasil dibuka.
Tekan 'q' pada jendela webcam untuk berhenti.

     HASIL VERIFIKASI WEBCAM
Total frame terbaca       : 258
Frame dengan tangan       : 229
Frame tanpa tangan        : 29
Landmark/frame terakhir   : 21
Fitur/frame terakhir      : 63

✓ Webcam berhasil mendeteksi tangan.
✓ Landmark tangan berhasil diperoleh.
✓ Jumlah landmark sesuai: 21.
✓ Jumlah fitur sesuai: 63.


In [7]:
# ==========================================
# CELL 4: PEMBENTUKAN SEQUENCE REAL-TIME
# DAN PEMUATAN MODEL CRNN
# ==========================================

import os
import numpy as np
import tensorflow as tf

print("=" * 60)
print("     CELL 4 - REAL-TIME SEQUENCE → CRNN")
print("=" * 60)


# ==========================================
# 1. KONFIGURASI SEQUENCE
# ==========================================

SEQUENCE_LENGTH = 30
FEATURES_PER_FRAME = 63
NUM_CLASSES = 26

print("\nKonfigurasi CRNN:")
print(f"- Sequence length : {SEQUENCE_LENGTH}")
print(f"- Fitur/frame     : {FEATURES_PER_FRAME}")
print(f"- Jumlah kelas    : {NUM_CLASSES}")
print(f"- Expected input  : ({SEQUENCE_LENGTH}, {FEATURES_PER_FRAME})")


# ==========================================
# 2. LOAD MODEL CRNN HASIL TRAINING
# ==========================================

MODEL_CRNN_PATH = (
    "/home/zacx/Downloads/SKRIPSI/Aplikasi/modelling/hasil_modelling/rasio_80_20/best_sibi_crnn_model.keras"
)

if not os.path.exists(MODEL_CRNN_PATH):

    print("\n❌ MODEL CRNN TIDAK DITEMUKAN!")
    print(f"Periksa path:")
    print(MODEL_CRNN_PATH)

else:

    model_crnn = tf.keras.models.load_model(
        MODEL_CRNN_PATH
    )

    print("\n✓ Model CRNN berhasil dimuat.")

    print("\nInput model CRNN:")
    print(model_crnn.input_shape)

    print("Output model CRNN:")
    print(model_crnn.output_shape)


# ==========================================
# 3. BUFFER SEQUENCE
# ==========================================

sequence_buffer = []


# ==========================================
# 4. FUNGSI MENAMBAHKAN FRAME
# ==========================================

def add_frame_to_sequence(frame_features):

    """
    Menambahkan satu frame hasil ekstraksi
    landmark ke dalam buffer sequence.

    Input:
        frame_features → array dengan 63 fitur

    Output:
        None
    """

    global sequence_buffer

    # Pastikan bentuk frame benar
    frame_features = np.asarray(
        frame_features,
        dtype=np.float32
    )

    if frame_features.shape != (
        FEATURES_PER_FRAME,
    ):

        raise ValueError(
            f"Shape frame tidak sesuai. "
            f"Diperoleh {frame_features.shape}, "
            f"seharusnya ({FEATURES_PER_FRAME},)"
        )

    # Masukkan frame ke buffer
    sequence_buffer.append(
        frame_features
    )

    # Jika lebih dari 30 frame,
    # buang frame paling lama
    if len(sequence_buffer) > SEQUENCE_LENGTH:

        sequence_buffer.pop(0)


# ==========================================
# 5. FUNGSI MEMBENTUK INPUT CRNN
# ==========================================

def get_crnn_input():

    """
    Mengubah buffer menjadi input CRNN.

    Output:
        Array dengan shape:
        (1, 30, 63)
    """

    if len(sequence_buffer) < SEQUENCE_LENGTH:

        return None

    sequence = np.array(
        sequence_buffer,
        dtype=np.float32
    )

    # Verifikasi shape sequence
    if sequence.shape != (
        SEQUENCE_LENGTH,
        FEATURES_PER_FRAME
    ):

        raise ValueError(
            f"Shape sequence tidak sesuai: "
            f"{sequence.shape}"
        )

    # Tambahkan batch dimension
    sequence = np.expand_dims(
        sequence,
        axis=0
    )

    return sequence


# ==========================================
# 6. UJI DENGAN DATA DUMMY
# ==========================================

print("\n" + "-" * 60)
print("     VERIFIKASI PEMBENTUKAN SEQUENCE")
print("-" * 60)

# Reset buffer
sequence_buffer = []

# Simulasikan 30 frame
for i in range(SEQUENCE_LENGTH):

    dummy_frame = np.zeros(
        FEATURES_PER_FRAME,
        dtype=np.float32
    )

    add_frame_to_sequence(
        dummy_frame
    )


crnn_input = get_crnn_input()


print(
    f"Jumlah frame dalam buffer : "
    f"{len(sequence_buffer)}"
)

if crnn_input is not None:

    print(
        f"Shape input CRNN         : "
        f"{crnn_input.shape}"
    )

    if crnn_input.shape == (
        1,
        SEQUENCE_LENGTH,
        FEATURES_PER_FRAME
    ):

        print(
            "\n✓ SEQUENCE BERHASIL DIBENTUK."
        )

        print(
            "✓ Shape sesuai dengan kebutuhan CRNN:"
        )

        print(
            "  (1, 30, 63)"
        )

    else:

        print(
            "\n❌ Shape input tidak sesuai."
        )

else:

    print(
        "\n❌ Sequence belum mencapai "
        "30 frame."
    )


# ==========================================
# 7. BERSIHKAN BUFFER
# ==========================================

sequence_buffer = []


print("\n" + "=" * 60)
print("     CELL 4 SELESAI")
print("=" * 60)

     CELL 4 - REAL-TIME SEQUENCE → CRNN

Konfigurasi CRNN:
- Sequence length : 30
- Fitur/frame     : 63
- Jumlah kelas    : 26
- Expected input  : (30, 63)

✓ Model CRNN berhasil dimuat.

Input model CRNN:
(None, 30, 63)
Output model CRNN:
(None, 26)

------------------------------------------------------------
     VERIFIKASI PEMBENTUKAN SEQUENCE
------------------------------------------------------------
Jumlah frame dalam buffer : 30
Shape input CRNN         : (1, 30, 63)

✓ SEQUENCE BERHASIL DIBENTUK.
✓ Shape sesuai dengan kebutuhan CRNN:
  (1, 30, 63)

     CELL 4 SELESAI


In [8]:
# ==========================================
# CELL 5: REAL-TIME CRNN PREDICTION
# WEBCAM → LANDMARK → SEQUENCE → CRNN
# ==========================================

print("=" * 60)
print("     CELL 5 - REAL-TIME CRNN PREDICTION")
print("=" * 60)


# ==========================================
# 1. PEMETAAN KELAS SIBI
# ==========================================

class_names = [
    chr(ord('A') + i)
    for i in range(NUM_CLASSES)
]

print("Pemetaan kelas CRNN:")

for idx, nama in enumerate(class_names):
    print(f"{idx:2d} → {nama}")


# ==========================================
# 2. VARIABEL PENYIMPANAN HASIL PREDIKSI
# ==========================================

hasil_prediksi_crnn = []

sequence_buffer = []

frame_count = 0
prediction_count = 0


# ==========================================
# 3. INISIALISASI WEBCAM
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    print("❌ Webcam tidak dapat dibuka.")

else:

    print("✓ Webcam berhasil dibuka.")
    print("✓ Mulai melakukan prediksi CRNN...")
    print("✓ Tekan 'q' untuk berhenti.")


# ==========================================
# 4. LOOP REAL-TIME
# ==========================================

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:

        print("❌ Frame gagal dibaca.")
        break


    frame_count += 1


    # --------------------------------------
    # Mirror webcam
    # --------------------------------------

    frame = cv2.flip(frame, 1)


    # --------------------------------------
    # BGR → RGB
    # --------------------------------------

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # ======================================
    # 5. DETEKSI LANDMARK
    # ======================================

    results = webcam_landmarker.process(
        frame_rgb
    )


    if results.multi_hand_landmarks:

        hand_landmarks = (
            results.multi_hand_landmarks[0]
        )


        # ----------------------------------
        # Gambar landmark
        # ----------------------------------

        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )


        # ----------------------------------
        # Ekstraksi 63 fitur
        # ----------------------------------

        frame_features = []

        for lm in hand_landmarks.landmark:

            frame_features.extend([
                lm.x,
                lm.y,
                lm.z
            ])


        frame_features = np.array(
            frame_features,
            dtype=np.float32
        )


        # ==================================
        # 6. MASUKKAN FRAME KE BUFFER
        # ==================================

        sequence_buffer.append(
            frame_features
        )


        # Menjaga buffer maksimal 30 frame

        if len(sequence_buffer) > SEQUENCE_LENGTH:

            sequence_buffer.pop(0)


        # ==================================
        # 7. JIKA SUDAH 30 FRAME
        # ==================================

        if len(sequence_buffer) == SEQUENCE_LENGTH:

            input_sequence = np.array(
                sequence_buffer,
                dtype=np.float32
            )


            # Tambahkan batch dimension
            input_sequence = np.expand_dims(
                input_sequence,
                axis=0
            )


            # ==================================
            # 8. PREDIKSI CRNN
            # ==================================

            prediction = model_crnn.predict(
                input_sequence,
                verbose=0
            )


            # Ambil probabilitas tertinggi
            predicted_index = np.argmax(
                prediction[0]
            )


            confidence = prediction[0][
                predicted_index
            ]


            predicted_class = class_names[
                predicted_index
            ]


            prediction_count += 1


            # ==================================
            # 9. SIMPAN HASIL PREDIKSI
            # ==================================

            hasil_prediksi_crnn.append(
                predicted_class
            )


            # ==================================
            # 10. TAMPILKAN HASIL DI WEBCAM
            # ==================================

            cv2.putText(
                frame,
                f"Prediksi : {predicted_class}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"Confidence : {confidence:.2%}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"Sequence : {len(sequence_buffer)}/30",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )


        else:

            cv2.putText(
                frame,
                f"Sequence : {len(sequence_buffer)}/30",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )


    else:

        cv2.putText(
            frame,
            "Tangan tidak terdeteksi",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )


    # ======================================
    # 11. INFORMASI FRAME
    # ======================================

    cv2.putText(
        frame,
        f"Frame : {frame_count}",
        (20, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ======================================
    # 12. TAMPILKAN WEBCAM
    # ======================================

    cv2.imshow(
        "SIBI - Real-Time CRNN Prediction",
        frame
    )


    # ======================================
    # 13. KELUAR
    # ======================================

    if cv2.waitKey(1) & 0xFF == ord('q'):

        break


# ==========================================
# 14. CLEANUP
# ==========================================

cap.release()
cv2.destroyAllWindows()


# ==========================================
# 15. HASIL AKHIR PREDIKSI CRNN
# ==========================================

print("\n" + "=" * 60)
print("     HASIL REAL-TIME CRNN")
print("=" * 60)

print(f"Total frame webcam       : {frame_count}")
print(f"Total prediksi CRNN      : {prediction_count}")
print(f"Total hasil tersimpan    : {len(hasil_prediksi_crnn)}")

print("\nBeberapa hasil prediksi:")
print(
    hasil_prediksi_crnn[:50]
)

print("=" * 60)

     CELL 5 - REAL-TIME CRNN PREDICTION
Pemetaan kelas CRNN:
 0 → A
 1 → B
 2 → C
 3 → D
 4 → E
 5 → F
 6 → G
 7 → H
 8 → I
 9 → J
10 → K
11 → L
12 → M
13 → N
14 → O
15 → P
16 → Q
17 → R
18 → S
19 → T
20 → U
21 → V
22 → W
23 → X
24 → Y
25 → Z
✓ Webcam berhasil dibuka.
✓ Mulai melakukan prediksi CRNN...
✓ Tekan 'q' untuk berhenti.

     HASIL REAL-TIME CRNN
Total frame webcam       : 354
Total prediksi CRNN      : 295
Total hasil tersimpan    : 295

Beberapa hasil prediksi:
['B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'K', 'K', 'K', 'K', 'K', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D', 'D']


In [9]:
# ==========================================
# CELL 6: STABILISASI OUTPUT CRNN
# PREDIKSI FRAME-LEVEL → URUTAN GESTURE
# ==========================================

print("=" * 60)
print("     CELL 6 - STABILISASI OUTPUT CRNN")
print("=" * 60)


# ==========================================
# 1. KONFIGURASI STABILISASI
# ==========================================

CONFIDENCE_THRESHOLD = 0.70

# Minimal jumlah prediksi yang sama
# secara berturut-turut sebelum dianggap
# sebagai gesture baru.
MIN_STABLE_PREDICTIONS = 5

print("Konfigurasi:")
print(
    f"- Confidence threshold : "
    f"{CONFIDENCE_THRESHOLD:.0%}"
)

print(
    f"- Minimal prediksi stabil : "
    f"{MIN_STABLE_PREDICTIONS}"
)


# ==========================================
# 2. VARIABEL STABILISASI
# ==========================================

sequence_buffer = []

prediksi_sebelumnya = None

jumlah_prediksi_sama = 0

gesture_stabil = None

hasil_gesture = []

total_frame = 0
total_prediksi = 0


# ==========================================
# 3. INISIALISASI WEBCAM
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    print("❌ Webcam tidak dapat dibuka.")

else:

    print("✓ Webcam berhasil dibuka.")
    print("✓ Stabilisasi output CRNN aktif.")
    print("✓ Tekan 'q' untuk berhenti.")


# ==========================================
# 4. LOOP REAL-TIME
# ==========================================

while cap.isOpened():

    ret, frame = cap.read()

    if not ret:

        print("❌ Frame gagal dibaca.")
        break


    total_frame += 1


    # --------------------------------------
    # Mirror webcam
    # --------------------------------------

    frame = cv2.flip(frame, 1)


    # --------------------------------------
    # BGR → RGB
    # --------------------------------------

    frame_rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # ======================================
    # 5. DETEKSI LANDMARK
    # ======================================

    results = webcam_landmarker.process(
        frame_rgb
    )


    if results.multi_hand_landmarks:

        hand_landmarks = (
            results.multi_hand_landmarks[0]
        )


        # ----------------------------------
        # Gambar landmark
        # ----------------------------------

        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )


        # ==================================
        # 6. EKSTRAKSI 63 FITUR
        # ==================================

        frame_features = []

        for lm in hand_landmarks.landmark:

            frame_features.extend([
                lm.x,
                lm.y,
                lm.z
            ])


        frame_features = np.array(
            frame_features,
            dtype=np.float32
        )


        # ==================================
        # 7. MASUKKAN KE BUFFER
        # ==================================

        sequence_buffer.append(
            frame_features
        )


        if len(sequence_buffer) > SEQUENCE_LENGTH:

            sequence_buffer.pop(0)


        # ==================================
        # 8. PREDIKSI JIKA 30 FRAME TERSEDIA
        # ==================================

        if len(sequence_buffer) == SEQUENCE_LENGTH:

            input_sequence = np.array(
                sequence_buffer,
                dtype=np.float32
            )


            input_sequence = np.expand_dims(
                input_sequence,
                axis=0
            )


            # --------------------------------
            # Prediksi CRNN
            # --------------------------------

            prediction = model_crnn.predict(
                input_sequence,
                verbose=0
            )


            predicted_index = np.argmax(
                prediction[0]
            )


            confidence = prediction[0][
                predicted_index
            ]


            predicted_class = class_names[
                predicted_index
            ]


            total_prediksi += 1


            # ==================================
            # 9. FILTER CONFIDENCE
            # ==================================

            if confidence >= CONFIDENCE_THRESHOLD:

                # --------------------------------
                # Jika prediksi sama dengan
                # prediksi sebelumnya
                # --------------------------------

                if predicted_class == prediksi_sebelumnya:

                    jumlah_prediksi_sama += 1

                else:

                    prediksi_sebelumnya = (
                        predicted_class
                    )

                    jumlah_prediksi_sama = 1


                # ==================================
                # 10. CEK APAKAH SUDAH STABIL
                # ==================================

                if (
                    jumlah_prediksi_sama
                    >= MIN_STABLE_PREDICTIONS
                ):

                    # Hanya simpan jika berbeda
                    # dari gesture stabil sebelumnya

                    if predicted_class != gesture_stabil:

                        gesture_stabil = (
                            predicted_class
                        )

                        hasil_gesture.append(
                            gesture_stabil
                        )

                        print(
                            f"✓ Gesture baru : "
                            f"{gesture_stabil} "
                            f"("
                            f"{confidence:.2%}"
                            f")"
                        )


            # ==================================
            # 11. INFORMASI REAL-TIME
            # ==================================

            cv2.putText(
                frame,
                f"Prediksi : {predicted_class}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (0, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"Confidence : {confidence:.2%}",
                (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )


            cv2.putText(
                frame,
                f"Stabil : {gesture_stabil}",
                (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )


            cv2.putText(
                frame,
                f"Stable Count : "
                f"{jumlah_prediksi_sama}/"
                f"{MIN_STABLE_PREDICTIONS}",
                (20, 160),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )


        else:

            cv2.putText(
                frame,
                f"Sequence : "
                f"{len(sequence_buffer)}/"
                f"{SEQUENCE_LENGTH}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )


    else:

        cv2.putText(
            frame,
            "Tangan tidak terdeteksi",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )


    # ======================================
    # 12. INFORMASI HASIL GESTURE
    # ======================================

    teks_hasil = " ".join(
        hasil_gesture[-10:]
    )

    cv2.putText(
        frame,
        f"Hasil : {teks_hasil}",
        (20, 200),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ======================================
    # 13. INFORMASI FRAME
    # ======================================

    cv2.putText(
        frame,
        f"Frame : {total_frame}",
        (20, 235),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )


    # ======================================
    # 14. TAMPILKAN WEBCAM
    # ======================================

    cv2.imshow(
        "SIBI - CRNN Gesture Stabilization",
        frame
    )


    # ======================================
    # 15. KELUAR
    # ======================================

    if cv2.waitKey(1) & 0xFF == ord('q'):

        break


# ==========================================
# 16. CLEANUP
# ==========================================

cap.release()
cv2.destroyAllWindows()


# ==========================================
# 17. HASIL AKHIR
# ==========================================

print("\n" + "=" * 60)
print("     HASIL STABILISASI CRNN")
print("=" * 60)

print(
    f"Total frame webcam       : "
    f"{total_frame}"
)

print(
    f"Total prediksi CRNN      : "
    f"{total_prediksi}"
)

print(
    f"Jumlah gesture stabil    : "
    f"{len(hasil_gesture)}"
)

print(
    "\nUrutan gesture hasil stabilisasi:"
)

print(
    hasil_gesture
)

print("=" * 60)

     CELL 6 - STABILISASI OUTPUT CRNN
Konfigurasi:
- Confidence threshold : 70%
- Minimal prediksi stabil : 5
✓ Webcam berhasil dibuka.
✓ Stabilisasi output CRNN aktif.
✓ Tekan 'q' untuk berhenti.
✓ Gesture baru : B (99.71%)
✓ Gesture baru : J (92.57%)
✓ Gesture baru : B (95.54%)

     HASIL STABILISASI CRNN
Total frame webcam       : 422
Total prediksi CRNN      : 321
Jumlah gesture stabil    : 3

Urutan gesture hasil stabilisasi:
['B', 'J', 'B']


In [13]:
# ==========================================
# CELL 7
# INTEGRASI CRNN → SEGMENTASI → RNN
# ==========================================

print("=" * 60)
print("     CELL 7 - INTEGRASI CRNN → SEGMENTASI → RNN")
print("=" * 60)


# ==========================================
# 1. KONFIGURASI
# ==========================================

MAX_SEQUENCE_LENGTH = 10

print("\n[1] Konfigurasi RNN")
print(f"- Max sequence length : {MAX_SEQUENCE_LENGTH}")


# ==========================================
# 2. CEK MODEL RNN
# ==========================================

print("\n[2] Memuat model RNN...")

try:
    model_rnn
    print("✓ Model RNN sudah tersedia di memory.")
except NameError:
    print("⚠ Model RNN belum tersedia di memory.")

    # Jika pada Cell sebelumnya sudah ada path model,
    # gunakan path tersebut.
    if 'MODEL_RNN_PATH' in globals():

        print(f"✓ Path model RNN ditemukan:")
        print(f"  {MODEL_RNN_PATH}")

        model_rnn = tf.keras.models.load_model(
            MODEL_RNN_PATH
        )

        print("✓ Model RNN berhasil dimuat.")

    else:
        raise NameError(
            "MODEL_RNN_PATH tidak ditemukan. "
            "Silakan definisikan path model RNN terlebih dahulu."
        )


# ==========================================
# 3. CEK TOKENIZER
# ==========================================

print("\n[3] Memeriksa tokenizer...")

if 'input_tokenizer' in globals():
    print("✓ Input tokenizer tersedia.")
else:
    print("⚠ Input tokenizer tidak ditemukan.")

if 'target_tokenizer' in globals():
    print("✓ Target tokenizer tersedia.")
else:
    print("⚠ Target tokenizer tidak ditemukan.")


# ==========================================
# 4. AMBIL HASIL SEGMENTASI DARI CRNN
# ==========================================

print("\n[4] Mengambil hasil segmentasi CRNN...")

if 'hasil_gesture' in globals():

    gesture_sequence = list(hasil_gesture)

elif 'hasil_prediksi_crnn' in globals():

    # fallback jika hasil_gesture tidak tersedia
    gesture_sequence = list(
        hasil_prediksi_crnn
    )

else:

    raise ValueError(
        "Tidak ditemukan hasil prediksi CRNN."
    )


print(f"✓ Jumlah gesture : {len(gesture_sequence)}")
print(f"✓ Gesture        : {gesture_sequence}")


# ==========================================
# 5. VALIDASI HASIL SEGMENTASI
# ==========================================

print("\n[5] Validasi sequence gesture...")

if len(gesture_sequence) == 0:

    print("⚠ Sequence gesture masih kosong.")

else:

    print("✓ Sequence gesture tersedia.")

    if len(gesture_sequence) > MAX_SEQUENCE_LENGTH:

        print(
            f"⚠ Sequence melebihi batas "
            f"{MAX_SEQUENCE_LENGTH}."
        )

        gesture_sequence = (
            gesture_sequence[-MAX_SEQUENCE_LENGTH:]
        )

        print(
            f"✓ Sequence dipotong menjadi "
            f"{len(gesture_sequence)} gesture."
        )

    else:

        print(
            f"✓ Panjang sequence = "
            f"{len(gesture_sequence)}"
        )


# ==========================================
# 6. KONVERSI GESTURE → TOKEN
# ==========================================

print("\n[6] Mengubah gesture menjadi token RNN...")

if 'input_tokenizer' in globals():

    try:

        gesture_text = " ".join(
            gesture_sequence
        )

        print(f"Input teks ke tokenizer:")
        print(f"  {gesture_text}")

        token_sequence = (
            input_tokenizer.texts_to_sequences(
                [gesture_text]
            )
        )

        print(
            f"✓ Token berhasil dibuat:"
        )
        print(
            f"  {token_sequence}"
        )

    except Exception as e:

        print(
            f"⚠ Tokenisasi gagal: {e}"
        )

        token_sequence = [[]]

else:

    print(
        "⚠ input_tokenizer tidak tersedia."
    )

    token_sequence = [[]]


# ==========================================
# 7. PADDING SEQUENCE
# ==========================================

print("\n[7] Melakukan padding sequence...")

if len(token_sequence[0]) > 0:

    rnn_input = tf.keras.preprocessing.sequence.pad_sequences(
        token_sequence,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding='post',
        truncating='post'
    )

    print(
        f"✓ Input RNN berhasil dibuat."
    )

    print(
        f"Shape input RNN : {rnn_input.shape}"
    )

else:

    rnn_input = None

    print(
        "⚠ Tidak dapat membuat input RNN "
        "karena token sequence kosong."
    )


# ==========================================
# 8. PREDIKSI RNN
# ==========================================

print("\n[8] Prediksi RNN...")

if rnn_input is not None:

    try:

        rnn_prediction = model_rnn.predict(
            rnn_input,
            verbose=0
        )

        print(
            "✓ Prediksi RNN berhasil dilakukan."
        )

        print(
            f"Shape output RNN : "
            f"{rnn_prediction.shape}"
        )

    except Exception as e:

        print(
            f"❌ Prediksi RNN gagal:"
        )

        print(e)

        rnn_prediction = None

else:

    rnn_prediction = None


# ==========================================
# 9. DECODE OUTPUT RNN
# ==========================================

print("\n[9] Decode hasil RNN...")

hasil_rnn = None

if (
    rnn_prediction is not None
    and 'target_tokenizer' in globals()
):

    try:

        # Ambil indeks token dengan probabilitas
        # terbesar
        predicted_ids = np.argmax(
            rnn_prediction,
            axis=-1
        )

        print(
            f"✓ Indeks prediksi berhasil diperoleh."
        )

        print(
            f"Shape indeks : "
            f"{predicted_ids.shape}"
        )

        # Konversi indeks → kata
        hasil_rnn = (
            target_tokenizer.sequences_to_texts(
                predicted_ids
            )
        )

        print(
            "✓ Hasil RNN berhasil di-decode."
        )

        print(
            f"Hasil RNN : {hasil_rnn}"
        )

    except Exception as e:

        print(
            f"⚠ Decode RNN gagal:"
        )

        print(e)

else:

    print(
        "⚠ Output RNN belum dapat di-decode."
    )


# ==========================================
# 10. RINGKASAN PIPELINE
# ==========================================

print("\n")
print("-" * 60)
print("     RINGKASAN PIPELINE")
print("-" * 60)

print(
    f"CRNN prediction : "
    f"{len(hasil_prediksi_crnn) if 'hasil_prediksi_crnn' in globals() else 0}"
)

print(
    f"Gesture hasil segmentasi : "
    f"{gesture_sequence}"
)

print(
    f"Input RNN : "
    f"{rnn_input.shape if rnn_input is not None else None}"
)

print(
    f"Hasil RNN : "
    f"{hasil_rnn}"
)

print("\n✓ CELL 7 SELESAI")
print("=" * 60)

     CELL 7 - INTEGRASI CRNN → SEGMENTASI → RNN

[1] Konfigurasi RNN
- Max sequence length : 10

[2] Memuat model RNN...
✓ Model RNN sudah tersedia di memory.

[3] Memeriksa tokenizer...
⚠ Input tokenizer tidak ditemukan.
⚠ Target tokenizer tidak ditemukan.

[4] Mengambil hasil segmentasi CRNN...
✓ Jumlah gesture : 3
✓ Gesture        : ['B', 'J', 'B']

[5] Validasi sequence gesture...
✓ Sequence gesture tersedia.
✓ Panjang sequence = 3

[6] Mengubah gesture menjadi token RNN...
⚠ input_tokenizer tidak tersedia.

[7] Melakukan padding sequence...
⚠ Tidak dapat membuat input RNN karena token sequence kosong.

[8] Prediksi RNN...

[9] Decode hasil RNN...
⚠ Output RNN belum dapat di-decode.


------------------------------------------------------------
     RINGKASAN PIPELINE
------------------------------------------------------------
CRNN prediction : 295
Gesture hasil segmentasi : ['B', 'J', 'B']
Input RNN : None
Hasil RNN : None

✓ CELL 7 SELESAI


In [ ]:
# ==========================================
# CELL 7A
# IDENTIFIKASI ARTEFAK MODEL RNN
# ==========================================

import os

print("=" * 60)
print("     CELL 7A - IDENTIFIKASI ARTEFAK RNN")
print("=" * 60)


# ==========================================
# 1. CEK PATH MODEL RNN
# ==========================================

print("\n[1] Informasi model RNN")

if '/home/zacx/Downloads/SKRIPSI/Aplikasi/hasil_modelling_rnn/rnn_suku_kata_model.keras' in globals():

    print("✓ MODEL_RNN_PATH ditemukan:")
    print('/home/zacx/Downloads/SKRIPSI/Aplikasi/hasil_modelling_rnn/rnn_suku_kata_model.keras')

    rnn_dir = os.path.dirname(
        '/home/zacx/Downloads/SKRIPSI/Aplikasi/hasil_modelling_rnn/rnn_suku_kata_model.keras'
    )

    print("\nFolder model RNN:")
    print(rnn_dir)

else:

    print("⚠ MODEL_RNN_PATH tidak ditemukan.")

    rnn_dir = None


# ==========================================
# 2. CEK ISI FOLDER MODEL RNN
# ==========================================

if rnn_dir is not None and os.path.exists(rnn_dir):

    print("\n[2] Isi folder model RNN")
    print("-" * 60)

    for root, dirs, files in os.walk(rnn_dir):

        level = root.replace(
            rnn_dir, ''
        ).count(os.sep)

        indent = '    ' * level

        print(
            f"{indent}{os.path.basename(root)}/"
        )

        subindent = '    ' * (level + 1)

        for file in files:

            print(
                f"{subindent}{file}"
            )

else:

    print(
        "⚠ Folder model RNN tidak ditemukan."
    )


# ==========================================
# 3. CARI FILE TOKENIZER
# ==========================================

print("\n[3] Mencari file tokenizer")

kata_kunci = [
    "token",
    "tokenizer",
    "vocab",
    "dictionary",
    "word_index"
]

file_tokenizer = []


if rnn_dir is not None:

    for root, dirs, files in os.walk(rnn_dir):

        for file in files:

            nama_lower = file.lower()

            if any(
                keyword in nama_lower
                for keyword in kata_kunci
            ):

                path_file = os.path.join(
                    root,
                    file
                )

                file_tokenizer.append(
                    path_file
                )


if len(file_tokenizer) > 0:

    print(
        f"✓ Ditemukan {len(file_tokenizer)} "
        "file yang berpotensi berkaitan dengan tokenizer:"
    )

    for path in file_tokenizer:

        print(
            f"  → {path}"
        )

else:

    print(
        "⚠ Tidak ditemukan file tokenizer "
        "berdasarkan nama file."
    )


# ==========================================
# 4. CEK VARIABEL TOKENIZER DI MEMORY
# ==========================================

print("\n[4] Variabel tokenizer di memory")

variabel_tokenizer = [
    nama
    for nama in list(globals().keys())
    if "token" in nama.lower()
]


if len(variabel_tokenizer) > 0:

    for nama in variabel_tokenizer:

        try:

            nilai = globals()[nama]

            print(
                f"✓ {nama} "
                f"→ {type(nilai).__name__}"
            )

        except Exception:

            pass

else:

    print(
        "⚠ Tidak ada variabel tokenizer "
        "di memory."
    )


# ==========================================
# 5. RINGKASAN
# ==========================================

print("\n")
print("=" * 60)
print("     RINGKASAN CELL 7A")
print("=" * 60)

print(
    f"Folder RNN ditemukan : "
    f"{rnn_dir is not None and os.path.exists(rnn_dir)}"
)

print(
    f"File kandidat tokenizer : "
    f"{len(file_tokenizer)}"
)

print(
    f"Variabel tokenizer : "
    f"{len(variabel_tokenizer)}"
)

print("=" * 60)
print("✓ CELL 7A SELESAI")
print("=" * 60)

     CELL 7A - IDENTIFIKASI ARTEFAK RNN

[1] Informasi model RNN
⚠ MODEL_RNN_PATH tidak ditemukan.
⚠ Folder model RNN tidak ditemukan.

[3] Mencari file tokenizer
⚠ Tidak ditemukan file tokenizer berdasarkan nama file.

[4] Variabel tokenizer di memory
✓ TOKENIZER_INPUT_PATH → str
✓ TOKENIZER_TARGET_PATH → str
✓ tokenizer_in → Tokenizer
✓ tokenizer_out → Tokenizer
✓ token_sequence → list
✓ file_tokenizer → list


     RINGKASAN CELL 7A
Folder RNN ditemukan : False
File kandidat tokenizer : 0
Variabel tokenizer : 6
✓ CELL 7A SELESAI


In [15]:
# ==========================================
# CELL 7B
# VERIFIKASI TOKENIZER DAN INPUT RNN
# ==========================================

print("=" * 60)
print("     CELL 7B - VERIFIKASI TOKENIZER RNN")
print("=" * 60)


# ==========================================
# 1. CEK TOKENIZER INPUT
# ==========================================

print("\n[1] Input tokenizer")

if 'tokenizer_in' in globals():

    print("✓ tokenizer_in tersedia.")
    print(f"  Tipe : {type(tokenizer_in).__name__}")

    try:
        print(
            f"  Jumlah vocabulary : "
            f"{len(tokenizer_in.word_index)}"
        )
    except Exception:
        print("  Vocabulary tidak dapat dibaca.")

else:

    raise NameError(
        "tokenizer_in tidak ditemukan."
    )


# ==========================================
# 2. CEK TOKENIZER TARGET
# ==========================================

print("\n[2] Target tokenizer")

if 'tokenizer_out' in globals():

    print("✓ tokenizer_out tersedia.")
    print(f"  Tipe : {type(tokenizer_out).__name__}")

    try:
        print(
            f"  Jumlah vocabulary : "
            f"{len(tokenizer_out.word_index)}"
        )
    except Exception:
        print("  Vocabulary tidak dapat dibaca.")

else:

    raise NameError(
        "tokenizer_out tidak ditemukan."
    )


# ==========================================
# 3. CEK HASIL GESTURE DARI CELL 6
# ==========================================

print("\n[3] Hasil segmentasi CRNN")

if 'hasil_gesture' in globals():

    gesture_sequence = list(
        hasil_gesture
    )

    print("✓ hasil_gesture tersedia.")
    print(
        f"  Jumlah gesture : "
        f"{len(gesture_sequence)}"
    )
    print(
        f"  Gesture        : "
        f"{gesture_sequence}"
    )

else:

    raise NameError(
        "hasil_gesture tidak ditemukan."
    )


# ==========================================
# 4. KONVERSI GESTURE → TEKS
# ==========================================

print("\n[4] Konversi gesture menjadi teks")

gesture_text = " ".join(
    gesture_sequence
)

print(
    f"✓ Input teks untuk tokenizer:"
)
print(
    f"  '{gesture_text}'"
)


# ==========================================
# 5. KONVERSI TEKS → TOKEN
# ==========================================

print("\n[5] Tokenisasi input")

token_sequence = tokenizer_in.texts_to_sequences(
    [gesture_text]
)

print(
    f"✓ Token sequence berhasil dibuat:"
)

print(
    f"  {token_sequence}"
)


# ==========================================
# 6. CEK TOKEN YANG TIDAK DIKENALI
# ==========================================

print("\n[6] Pemeriksaan vocabulary")

tokenizer_word_index = (
    tokenizer_in.word_index
)

gesture_unknown = []

for gesture in gesture_sequence:

    if gesture.lower() not in tokenizer_word_index:

        gesture_unknown.append(
            gesture
        )


if len(gesture_unknown) == 0:

    print(
        "✓ Semua gesture ditemukan "
        "dalam vocabulary tokenizer."
    )

else:

    print(
        "⚠ Gesture berikut tidak ditemukan "
        "dalam vocabulary:"
    )

    print(
        f"  {gesture_unknown}"
    )


# ==========================================
# 7. PADDING
# ==========================================

print("\n[7] Pembentukan input RNN")

MAX_SEQUENCE_LENGTH = 10

rnn_input = tf.keras.preprocessing.sequence.pad_sequences(
    token_sequence,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding='post',
    truncating='post'
)

print(
    "✓ Padding berhasil dilakukan."
)

print(
    f"  Max sequence length : "
    f"{MAX_SEQUENCE_LENGTH}"
)

print(
    f"  Shape input RNN    : "
    f"{rnn_input.shape}"
)

print(
    f"  Input RNN          : "
    f"{rnn_input}"
)


# ==========================================
# 8. CEK INPUT MODEL RNN
# ==========================================

print("\n[8] Verifikasi shape model RNN")

print(
    f"  Model input : "
    f"{model_rnn.input_shape}"
)

print(
    f"  Input kita  : "
    f"{rnn_input.shape}"
)


# ==========================================
# 9. HASIL VERIFIKASI
# ==========================================

print("\n")
print("=" * 60)
print("     HASIL VERIFIKASI CELL 7B")
print("=" * 60)

print(
    "✓ tokenizer_in tersedia"
)

print(
    "✓ tokenizer_out tersedia"
)

print(
    f"✓ Gesture : {gesture_sequence}"
)

print(
    f"✓ Token   : {token_sequence}"
)

print(
    f"✓ RNN input shape : {rnn_input.shape}"
)

print("=" * 60)
print("✓ CELL 7B SELESAI")
print("=" * 60)

     CELL 7B - VERIFIKASI TOKENIZER RNN

[1] Input tokenizer
✓ tokenizer_in tersedia.
  Tipe : Tokenizer
  Jumlah vocabulary : 929

[2] Target tokenizer
✓ tokenizer_out tersedia.
  Tipe : Tokenizer
  Jumlah vocabulary : 26

[3] Hasil segmentasi CRNN
✓ hasil_gesture tersedia.
  Jumlah gesture : 3
  Gesture        : ['B', 'J', 'B']

[4] Konversi gesture menjadi teks
✓ Input teks untuk tokenizer:
  'B J B'

[5] Tokenisasi input
✓ Token sequence berhasil dibuat:
  [[1, 1, 1]]

[6] Pemeriksaan vocabulary
⚠ Gesture berikut tidak ditemukan dalam vocabulary:
  ['B', 'J', 'B']

[7] Pembentukan input RNN
✓ Padding berhasil dilakukan.
  Max sequence length : 10
  Shape input RNN    : (1, 10)
  Input RNN          : [[1 1 1 0 0 0 0 0 0 0]]

[8] Verifikasi shape model RNN
  Model input : (None, 12)
  Input kita  : (1, 10)


     HASIL VERIFIKASI CELL 7B
✓ tokenizer_in tersedia
✓ tokenizer_out tersedia
✓ Gesture : ['B', 'J', 'B']
✓ Token   : [[1, 1, 1]]
✓ RNN input shape : (1, 10)
✓ CELL 7B SELESAI
